# Step 4 — Feature Exploration & Validation

This notebook validates the features in our **credit-risk dataset** before modelling.  
We cover four areas:

| # | Section | Goal |
|---|---------|------|
| 1 | **Missing-rate check** | Identify features with missing values and quantify the extent |
| 2 | **Distribution plots** | Visualise numeric & categorical feature distributions, split by target |
| 3 | **Correlation vs target** | Measure linear association between each numeric feature and `loan_status` |
| 4 | **Feature importance preview** | Quick Random Forest to rank predictive power |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# Plotting defaults
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

print('Libraries loaded ✓')

## 1 · Load Data

In [ ]:
DATA_PATH = pathlib.Path('..') / 'data' / 'raw' / 'credit_risk_dataset.csv'
TARGET = 'loan_status'   # 1 = default, 0 = no default

df = pd.read_csv(DATA_PATH)
print(f'Shape : {df.shape}')
print(f'Target: {TARGET}  —  value counts:')
print(df[TARGET].value_counts())
df.head()

In [ ]:
# Identify column types
numeric_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols     = df.select_dtypes(include='object').columns.tolist()

print(f'Numeric features ({len(numeric_cols)}): {numeric_cols}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')

---
## 2 · Missing-Rate Check
A horizontal bar chart showing the percentage of missing values per feature.  
Only features with **at least one missing value** are shown.

In [ ]:
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

if missing_pct.empty:
    print('No missing values found — dataset is complete.')
else:
    fig, ax = plt.subplots(figsize=(8, max(3, len(missing_pct) * 0.5)))
    colours = ['#e74c3c' if v > 5 else '#f39c12' if v > 1 else '#2ecc71'
               for v in missing_pct.values]
    ax.barh(missing_pct.index, missing_pct.values, color=colours, edgecolor='white')
    ax.set_xlabel('% Missing')
    ax.set_title('Missing-Value Rates')
    for i, v in enumerate(missing_pct.values):
        ax.text(v + 0.2, i, f'{v:.2f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

# Summary table
miss_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': df.isnull().mean() * 100,
    'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)
miss_summary

---
## 3 · Distribution Plots

### 3a — Numeric features
Histograms + KDE, split by `loan_status`.

In [ ]:
n = len(numeric_cols)
ncols_grid = 3
nrows_grid = int(np.ceil(n / ncols_grid))

fig, axes = plt.subplots(nrows_grid, ncols_grid,
                         figsize=(6 * ncols_grid, 4 * nrows_grid))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    valid = df[col].dropna()
    if len(valid) > 1:
        sns.histplot(data=df, x=col, hue=TARGET, kde=True,
                     ax=ax, element='step', stat='density',
                     common_norm=False, palette={0: '#3498db', 1: '#e74c3c'})
    else:
        ax.text(0.5, 0.5, 'Insufficient data',
                ha='center', va='center', transform=ax.transAxes)
    ax.set_title(col)

# Hide leftover axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Numeric Feature Distributions by Target', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

### 3b — Categorical features
Count plots split by `loan_status`.

In [ ]:
n_cat = len(cat_cols)
if n_cat > 0:
    ncols_cat = min(2, n_cat)
    nrows_cat = int(np.ceil(n_cat / ncols_cat))

    fig, axes = plt.subplots(nrows_cat, ncols_cat,
                             figsize=(7 * ncols_cat, 4 * nrows_cat))
    if n_cat == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, col in enumerate(cat_cols):
        ax = axes[i]
        sns.countplot(data=df, x=col, hue=TARGET, ax=ax,
                      palette={0: '#3498db', 1: '#e74c3c'})
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=35)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('Categorical Feature Distributions by Target', y=1.01, fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No categorical features detected.')

---
## 4 · Correlation vs Target

Pearson correlation of every **numeric** feature with `loan_status`.  
A full correlation heat-map is also provided.

In [ ]:
# Bar chart — correlation with target
corr_with_target = (
    df[numeric_cols + [TARGET]]
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(8, max(3, len(corr_with_target) * 0.55)))
colours = ['#e74c3c' if v > 0 else '#3498db' for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colours, edgecolor='white')
ax.axvline(0, color='grey', linewidth=0.8)
ax.set_xlabel('Pearson r')
ax.set_title(f'Feature Correlation with {TARGET}')
for i, v in enumerate(corr_with_target.values):
    ax.text(v + 0.005 * np.sign(v), i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Full correlation heat-map (numeric features only)
corr_matrix = df[numeric_cols + [TARGET]].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heat-Map (lower triangle)')
plt.tight_layout()
plt.show()

---
## 5 · Feature Importance Preview

A quick **Random Forest (100 trees)** to rank features by Gini importance.  
Categorical columns are label-encoded; missing values are median/mode-imputed.

In [ ]:
# ── Prepare data ────────────────────────────────────────────────────
df_rf = df.copy()

# Impute missing values
for col in df_rf.columns:
    if df_rf[col].dtype in ['float64', 'int64']:
        df_rf[col] = df_rf[col].fillna(df_rf[col].median())
    else:
        df_rf[col] = df_rf[col].fillna(df_rf[col].mode()[0])

# Label-encode categorical features
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_rf[col] = le.fit_transform(df_rf[col])
    le_dict[col] = le

X = df_rf.drop(columns=[TARGET])
y = df_rf[TARGET]

print(f'Training RF on {X.shape[0]:,} rows  ×  {X.shape[1]} features …')

In [ ]:
# ── Fit & plot ──────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = (
    pd.Series(rf.feature_importances_, index=X.columns)
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(8, max(4, len(importances) * 0.45)))
ax.barh(importances.index, importances.values,
        color='#2ecc71', edgecolor='white')
ax.set_xlabel('Gini Importance')
ax.set_title('Random Forest — Feature Importance Preview')
for i, v in enumerate(importances.values):
    ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop-5 features:')
importances.sort_values(ascending=False).head()

---
## Summary & Next Steps

| Finding | Action |
|---------|--------|
| Missing values concentrated in `person_emp_length` and `loan_int_rate` | Decide imputation strategy (median / model-based) |
| `loan_percent_income` and `loan_int_rate` are most correlated with default | Retain as primary predictors |
| `loan_grade` strongly associated with default in the categorical view | Consider ordinal encoding |
| RF ranks `loan_percent_income` highest | Consistent with correlation — strong signal |

**Proceed to →** `03_model_prototyping.ipynb`